In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn #neural network module
import torch.optim as optim #optimizer module
import torchvision.models as models#contains all models 
import torchvision.transforms as transforms#daata augmentation and normalising numbers
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader#loads imgages into batches of 32 each time and shuffles
from sklearn.metrics import classification_report , confusion_matrix#reports precision and recall and confusion matrix
from torchvision.transforms import v2
from torchvision import datasets
print('done')

done


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
train_transforms = transforms.Compose([v2.Resize((224,224)),#resize images to 224x224 pixels
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),#scales values from 0 to 1
                    v2.Normalize(mean=[0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])])

train_ds = datasets.ImageFolder(root='/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia',transform=train_transforms,
is_valid_file=lambda path: '__MACOSX' not in path and not os.path.basename(path).startswith('.'))#correct file path
dataLoader = DataLoader(train_ds,32,shuffle='true')#load minibatches
print(len(dataLoader))


test_ds = datasets.ImageFolder(root='/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test',transform=train_transforms,
                     is_valid_file=lambda path: '__MACOSX' not in path and not os.path.basename(path).startswith('.'))
test_loader = DataLoader(test_ds,32,shuffle='false')
print(len(test_loader))

366
20


In [4]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)#resnet model
for parameter in model.parameters():
    parameter.requires_grad = False#Freezes brackprop for fully connected layers
new_head = nn.Linear(in_features=512,out_features=2)
model.fc = new_head
model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [9]:
loss_criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(),lr=0.001)
print('done')


done


In [8]:
model.train()
total_epochs = 10
for epoch in range(total_epochs):
    epoch_loss = 0.0
    for images, labels in dataLoader:
        images = images.to(device='cuda')
        labels = labels.to(device='cuda')
        optimizer.zero_grad()
        predicted_logits = model(images)
        loss = loss_criterion(predicted_logits,labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    average_loss = epoch_loss / len(dataLoader)
    print("Epoch number", epoch, ": average loss-",average_loss)
    

KeyboardInterrupt: 

In [ ]:
model.eval()
all_preds = []
all_lables = []
with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device='cuda')
        labels = labels.to(device='cuda')
        outputs = model(images)
        preds = torch.argmax(outputs , dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(outputs.cpu().numpy())
confusion_matrix(all_labels,all_preds)
classes = ['normal','pneumonia']
classification_report(all_labels,all_preds,target_names=classes)